### Importing packages

In [1]:
import pandas as pd              
import numpy as np             

In [2]:
test = pd.read_csv(
    f"/home/users/mendrika/NCAST/Data/Dakar/data-test-dakar.csv"
)

train_full = pd.read_csv(
    f"/home/users/mendrika/NCAST/Data/Dakar/data-train-dakar.csv"
)

val   = train_full[train_full["year"] == 2019].copy()
train = train_full[train_full["year"] != 2019].copy()

In [3]:
feature_cols = [
    "year",
    "month",
    "day",
    "hour",
    "minute",

    "lat1", "lat2", "lat3",
    "lon1", "lon2", "lon3",

    "wp1", "wp2", "wp3",

    "size1", "size2", "size3",

    "d1", "d2", "d3",

    "mask1", "mask2", "mask3"
]

In [4]:
target_cols = [
    "Cb_dakar_t0",
    "Cb_dakar_t1",
    "Cb_dakar_t2",
    "Cb_dakar_t3",
    "Cb_dakar_t4",
    "Cb_dakar_t5",
    "Cb_dakar_t6"
]

### Choosing lead time

In [5]:
def create_mlp2t_dataset(df):

    # Create a copy to avoid modifying the original dataframe
    df = df.copy()

    # Create a datetime column from the timestamp components (everything at t0)
    df["time"] = pd.to_datetime({
        "year": df["year"],
        "month": df["month"],
        "day": df["day"],
        "hour": df["hour"],
        "minute": df["minute"]
    })

    # Select previous-time features together with the timestamp
    prev_df = df[["time"] + feature_cols].copy()

    # Shift timestamps forward by 1 hour
    # Features originally at t-1h will now align with rows at t0
    prev_df["time"] = prev_df["time"] + pd.Timedelta(hours=1)

    # Rename columns to indicate features from t-1h
    prev_df = prev_df.rename(
        columns={
            col: f"{col}_tm1h"
            for col in feature_cols
        }
    )

    # Merge current-time features with t-1h features (primary key is time here for merging)
    merged = df.merge(
        prev_df,
        on="time",
        how="inner"
    )

    return merged

In [6]:
train_2t = create_mlp2t_dataset(train)
val_2t   = create_mlp2t_dataset(val)
test_2t  = create_mlp2t_dataset(test)

# Preprocessing

In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler

In [8]:
def prepare_2t_data(
    train_2t,
    val_2t,
    test_2t,
    lead_time,
    n_cores=3,
):

    # Select the prediction target for the requested lead time
    target_header = f"Cb_dakar_t{lead_time}"

    # Keep all input columns except targets and the datetime column
    input_headers = [
        col
        for col in train_2t.columns
        if not col.startswith("Cb_")
        and col != "time"
    ]

    # Split inputs and targets
    X_train = train_2t[input_headers].copy()
    X_val   = val_2t[input_headers].copy()
    X_test  = test_2t[input_headers].copy()

    y_train = train_2t[target_header].copy()
    y_val   = val_2t[target_header].copy()
    y_test  = test_2t[target_header].copy()

    # Features requiring log scaling
    cols_to_log = []

    for prefix in ["size", "wp", "d"]:

        # Current-time features
        cols_to_log.extend([
            f"{prefix}{i}"
            for i in range(1, n_cores + 1)
        ])

        # Previous-time features
        cols_to_log.extend([
            f"{prefix}{i}_tm1h"
            for i in range(1, n_cores + 1)
        ])

    # Apply log1p transformation
    for col in cols_to_log:

        X_train[col] = np.log1p(X_train[col])
        X_val[col]   = np.log1p(X_val[col])
        X_test[col]  = np.log1p(X_test[col])

    # Define groups of features for each storm core
    core_groups = {}

    for i in range(1, n_cores + 1):

        core_groups[i] = [
            f"lat{i}",
            f"lon{i}",
            f"wp{i}",
            f"size{i}",
            f"d{i}",
        ]

    # Create copies that will contain scaled data
    X_train_scaled = X_train.copy()
    X_val_scaled   = X_val.copy()
    X_test_scaled  = X_test.copy()

    # Store fitted scalers
    scalers = {}

    # Scale storm-core features
    for i, cols in core_groups.items():

        # Only fit on real storm cores
        mask = X_train[f"mask{i}"] == 1

        scaler = StandardScaler()

        scaler.fit(
            X_train.loc[mask, cols]
        )

        scalers[i] = scaler

        # Corresponding t-1h columns
        tm1_cols = [
            f"{col}_tm1h"
            for col in cols
        ]

        # Scale current-time features
        X_train_scaled.loc[:, cols] = (               
            scaler.transform(X_train[cols])
        )

        X_val_scaled.loc[:, cols] = (
            scaler.transform(X_val[cols])
        )

        X_test_scaled.loc[:, cols] = (
            scaler.transform(X_test[cols])
        )

        # Copy previous-time features
        train_tm1 = X_train[tm1_cols].copy()
        val_tm1   = X_val[tm1_cols].copy()
        test_tm1  = X_test[tm1_cols].copy()

        # Rename so the scaler sees identical feature names
        train_tm1.columns = cols
        val_tm1.columns   = cols
        test_tm1.columns  = cols

        # Scale previous-time features using the same scaler
        X_train_scaled.loc[:, tm1_cols] = (
            scaler.transform(train_tm1)
        )

        X_val_scaled.loc[:, tm1_cols] = (
            scaler.transform(val_tm1)
        )

        X_test_scaled.loc[:, tm1_cols] = (
            scaler.transform(test_tm1)
        )

    # Temporal features at t0
    time_cols = [
        "year",
        "month",
        "day",
        "hour",
        "minute",
    ]

    # Temporal features at t-1h
    time_cols_tm1 = [
        f"{col}_tm1h"
        for col in time_cols
    ]

    # Convert to float before scaling
    X_train_scaled[time_cols] = (
        X_train_scaled[time_cols].astype(float)
    )

    X_val_scaled[time_cols] = (
        X_val_scaled[time_cols].astype(float)
    )

    X_test_scaled[time_cols] = (
        X_test_scaled[time_cols].astype(float)
    )

    X_train_scaled[time_cols_tm1] = (
        X_train_scaled[time_cols_tm1].astype(float)
    )

    X_val_scaled[time_cols_tm1] = (
        X_val_scaled[time_cols_tm1].astype(float)
    )

    X_test_scaled[time_cols_tm1] = (
        X_test_scaled[time_cols_tm1].astype(float)
    )

    # Fit scaler on temporal features
    time_scaler = StandardScaler()

    time_scaler.fit(
        X_train[time_cols]
    )

    # Scale current-time temporal features
    X_train_scaled.loc[:, time_cols] = (
        time_scaler.transform(X_train[time_cols])
    )

    X_val_scaled.loc[:, time_cols]   = (
        time_scaler.transform(X_val[time_cols])
    )

    X_test_scaled.loc[:, time_cols]  = (
        time_scaler.transform(X_test[time_cols])
    )

    # Copy t-1h temporal features
    train_tm1 = X_train[time_cols_tm1].copy()
    val_tm1   = X_val[time_cols_tm1].copy()
    test_tm1  = X_test[time_cols_tm1].copy()

    # Rename so feature names match the scaler input
    train_tm1.columns = time_cols
    val_tm1.columns   = time_cols
    test_tm1.columns  = time_cols

    # Scale t-1h temporal features
    X_train_scaled.loc[:, time_cols_tm1] = (
        time_scaler.transform(train_tm1)
    )

    X_val_scaled.loc[:, time_cols_tm1]   = (
        time_scaler.transform(val_tm1)
    )

    X_test_scaled.loc[:, time_cols_tm1]  = (
        time_scaler.transform(test_tm1)
    )

    return (
        X_train_scaled,
        X_val_scaled,
        X_test_scaled,
        y_train,
        y_val,
        y_test,
        scalers,
    )

In [9]:
X_train, X_val, X_test, y_train, y_val, y_test, scalers = prepare_2t_data(
    train_2t=train_2t,
    val_2t=val_2t,
    test_2t=test_2t,
    lead_time=1,
)

In [10]:
X_train

,year,month,day,hour,minute,lat1,lat2,lat3,lon1,lon2,...,wp3_tm1h,size1_tm1h,size2_tm1h,size3_tm1h,d1_tm1h,d2_tm1h,d3_tm1h,mask1_tm1h,mask2_tm1h,mask3_tm1h
0,1.247761,-1.511643,-1.690425,-1.531782,-1.341844,-1.663119,-1.585614,-2.329475,1.048681,1.106570,...,-22.491395,0.225856,-4.861156,-4.890168,1.258535,2.142076,2.288676,1,0,0
1,1.247761,-1.511643,-1.690425,-1.531782,-0.447364,-1.636555,-1.573586,-0.872385,0.995310,1.094254,...,-22.491395,0.111262,-4.861156,-4.890168,1.244463,2.001176,2.021220,1,0,0
2,1.247761,-1.511643,-1.690425,-1.531782,0.447117,-1.636555,-1.585526,4.372904,0.995310,1.094748,...,-22.491395,-0.042227,-4.861156,-4.890168,1.187734,1.787110,2.199390,1,0,0
3,1.247761,-1.511643,-1.690425,-1.531782,1.341597,-1.596649,-1.721085,-1.354611,0.915074,2.429140,...,-22.491395,-0.400673,-1.227394,-4.890168,1.189285,1.264102,2.339630,1,1,0
4,1.247761,-1.511643,-1.690425,-1.120485,0.447117,-1.605140,-2.818887,4.576917,0.377080,-3.667084,...,-22.491395,-1.111260,-4.861156,-4.890168,1.034488,1.845016,2.489078,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94273,-1.599079,1.217565,1.587730,0.798903,-0.447364,-0.899363,4.475841,-2.471420,0.775060,0.992742,...,-22.491395,0.791509,-4.861156,-4.890168,0.807398,1.763373,1.942690,1,0,0
94274,-1.599079,1.217565,1.587730,0.798903,0.447117,-0.779376,3.235179,-0.118672,0.649601,-4.276906,...,-22.491395,0.788300,-4.861156,-4.890168,0.793288,1.820155,1.899878,1,0,0
94275,-1.599079,1.217565,1.587730,0.798903,1.341597,-0.792045,3.555435,-1.437475,0.597453,-4.380900,...,-22.491395,0.734821,-4.861156,-4.890168,0.814444,1.760062,2.009502,1,0,0
94276,-1.599079,1.217565,1.587730,0.936002,-1.341844,-0.778437,3.607335,-1.902273,0.556997,-4.397997,...,-22.491395,0.625835,-4.861156,-4.890168,0.794392,1.870978,2.146840,1,0,0


In [11]:
X_val

,year,month,day,hour,minute,lat1,lat2,lat3,lon1,lon2,...,wp3_tm1h,size1_tm1h,size2_tm1h,size3_tm1h,d1_tm1h,d2_tm1h,d3_tm1h,mask1_tm1h,mask2_tm1h,mask3_tm1h
0,1.959471,-1.511643,-1.690425,0.524705,1.341597,-0.696089,4.325976,-2.368669,1.603410,-3.569407,...,-22.491395,-1.351971,-0.215697,-4.890168,1.004399,1.052229,1.871687,1,1,0
1,1.959471,-1.511643,-1.690425,0.661804,-1.341844,-0.709375,2.268397,4.099188,1.617072,-4.675975,...,-22.491395,0.401593,-4.861156,-4.890168,1.087191,1.845104,2.010375,1,0,0
2,1.959471,-1.511643,-1.690425,0.661804,-0.447364,-0.696313,-0.414102,-2.109489,1.629497,1.552414,...,-22.491395,0.719000,-4.861156,-4.890168,1.064591,1.840533,1.877251,1,0,0
3,1.959471,-1.511643,-1.690425,0.661804,0.447117,-0.696198,-0.635407,-0.563289,1.616450,2.949823,...,-22.491395,0.932809,-4.861156,-4.890168,1.049695,1.802691,2.309873,1,0,0
4,1.959471,-1.511643,-1.690425,0.661804,1.341597,-0.682870,-2.270504,4.032639,1.602740,-4.586345,...,-22.491395,1.088472,-4.861156,-4.890168,1.048319,1.951134,1.964211,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5657,1.959471,1.217565,1.587730,1.347299,1.341597,-1.720382,-2.706315,3.807589,1.713291,-3.800068,...,-22.491395,-1.251597,-4.861156,-4.890168,1.409596,2.022020,2.522215,1,0,0
5658,1.959471,1.217565,1.587730,1.484398,-1.341844,-1.720198,-2.804654,4.401316,1.687428,1.467071,...,-22.491395,0.387347,-4.861156,-4.890168,1.394728,2.198700,2.317277,1,0,0
5659,1.959471,1.217565,1.587730,1.484398,-0.447364,-1.706946,-2.201742,-1.461556,1.661036,2.139638,...,-22.491395,0.431934,-4.861156,-4.890168,1.394728,1.805982,1.998307,1,0,0
5660,1.959471,1.217565,1.587730,1.484398,0.447117,-1.667846,-2.451114,-1.705940,1.672423,-3.637961,...,-22.491395,0.437310,-4.861156,-4.890168,1.383324,2.033958,1.858468,1,0,0


In [12]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score


lead_time = 6

X_train, X_val, X_test, y_train, y_val, y_test, scalers = prepare_2t_data(
    train_2t=train_2t,
    val_2t=val_2t,
    test_2t=test_2t,
    lead_time=lead_time,
    n_cores=3,
)


mlp_2t = MLPClassifier(
    hidden_layer_sizes=(64, 16, 8),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
)


mlp_2t.fit(
    X_train,
    y_train,
)


y_prob = mlp_2t.predict_proba(
    X_test
)[:, 1]


auc = roc_auc_score(
    y_test,
    y_prob,
)

brier = brier_score_loss(
    y_test,
    y_prob,
)

auprc = average_precision_score(
    y_test,
    y_prob,
)


print(f"Lead time: {lead_time} h")
print(f"MLP-2T AUC: {auc:.4f}")
print(f"MLP-2T AUPRC: {auprc:.4f}")
print(f"MLP-2T Brier score: {brier:.4f}")

Lead time: 6 h
MLP-2T AUC: 0.7231
MLP-2T AUPRC: 0.0271
MLP-2T Brier score: 0.0123


In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

from sklearn.metrics import roc_auc_score

from itertools import product

import pandas as pd
import numpy as np


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


lead_time = 6


X_train, X_val, X_test, y_train, y_val, y_test, scalers = prepare_2t_data(
    train_2t=train_2t,
    val_2t=val_2t,
    test_2t=test_2t,
    lead_time=lead_time,
    n_cores=3,
)


X_train_tensor = torch.tensor(
    X_train.values,
    dtype=torch.float32,
)

X_val_tensor = torch.tensor(
    X_val.values,
    dtype=torch.float32,
)

X_test_tensor = torch.tensor(
    X_test.values,
    dtype=torch.float32,
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32,
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32,
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32,
)


class MLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_layers,
        dropout,
    ):

        super().__init__()

        layers = []

        prev_dim = input_dim

        for hidden_dim in hidden_layers:

            layers.append(
                nn.Linear(prev_dim, hidden_dim)
            )

            layers.append(
                nn.ReLU()
            )

            layers.append(
                nn.Dropout(dropout)
            )

            prev_dim = hidden_dim

        layers.append(
            nn.Linear(prev_dim, 1)
        )

        self.network = nn.Sequential(*layers)

    def forward(self, x):

        return self.network(x).squeeze(1)


hidden_layers_list = [
    (64, 32),
    (128, 64),
    (128, 64, 32),
]

learning_rates = [
    1e-2,
    1e-3,
]

dropouts = [
    0.1,
    0.2,
    0.3,
]

weight_decays = [
    1e-5,
    1e-4,
]

batch_sizes = [
    128,
    256,
]


results = []


for hidden_layers, lr, dropout, weight_decay, batch_size in product(
    hidden_layers_list,
    learning_rates,
    dropouts,
    weight_decays,
    batch_sizes,
):

    train_loader = DataLoader(
        TensorDataset(
            X_train_tensor,
            y_train_tensor,
        ),
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        TensorDataset(
            X_val_tensor,
            y_val_tensor,
        ),
        batch_size=batch_size,
        shuffle=False,
    )

    model = MLP(
        input_dim=X_train.shape[1],
        hidden_layers=hidden_layers,
        dropout=dropout,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    best_val_auc = 0

    for epoch in range(20):

        model.train()

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = criterion(
                logits,
                y_batch,
            )

            loss.backward()

            optimizer.step()

        model.eval()

        val_probs = []
        val_targets = []

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)

                logits = model(X_batch)

                probs = torch.sigmoid(logits)

                val_probs.extend(
                    probs.cpu().numpy()
                )

                val_targets.extend(
                    y_batch.numpy()
                )

        val_auc = roc_auc_score(
            val_targets,
            val_probs,
        )

        if val_auc > best_val_auc:

            best_val_auc = val_auc

    results.append({
        "hidden_layers": hidden_layers,
        "learning_rate": lr,
        "dropout": dropout,
        "weight_decay": weight_decay,
        "batch_size": batch_size,
        "val_auc": best_val_auc,
    })

    print(
        f"{hidden_layers} | "
        f"lr={lr} | "
        f"dropout={dropout} | "
        f"wd={weight_decay} | "
        f"batch={batch_size} | "
        f"AUC={best_val_auc:.4f}"
    )


results = pd.DataFrame(results)

results = results.sort_values(
    by="val_auc",
    ascending=False,
)

print(results.head())